In [1]:
import pandas as pd
import numpy as np
import itertools
import networkx as nx
import semopy
import itertools
import json
import os
from warnings import filterwarnings
filterwarnings("ignore")

In [2]:
path_controle = os.path.join("..", "dados", "raios_espectrais", "raios_espectrais_SitioIdadeSexo_controle.csv")
path_autismo = os.path.join("..", "dados", "raios_espectrais", "raios_espectrais_SitioIdadeSexo_autismo.csv")

path_controle_prep = os.path.join("..", "dados", "raios_espectrais", "raios_espectrais_SitioIdadeSexo_controle_prep.csv")
path_autismo_prep = os.path.join("..", "dados", "raios_espectrais", "raios_espectrais_SitioIdadeSexo_autismo_prep.csv")


salvar_formulas = os.path.join("..", "utils", "formulas_DAGS_5_conexos.json")
salvar_modelos_controle = os.path.join("..", "dados", "resultados", "resultados_empiricos", "modelos_controle.csv")
salvar_modelos_autismo = os.path.join("..", "dados", "resultados", "resultados_empiricos", "modelos_autismo.csv")

In [3]:
df_controle = pd.read_csv(path_controle_prep)
df_autismo = pd.read_csv(path_autismo_prep)

### Construção de Fórmulas de Modelos com 5 vértices conexos

In [4]:
def gerar_todas_dags_conexas_5v():
    vertices = ['Visual', 'Somatomotor', 'Default_mode', 'Cerebellar', 'Control']
    # Gera os 10 pares possíveis de ligações
    pares = list(itertools.combinations(vertices, 2))
    
    dags_validas = {}
    contador = 1
    
    # Testa todas as 59.049 combinações (3^10)
    # 0 = sem aresta, 1 = u->v, 2 = v->u
    for estados in itertools.product([0, 1, 2], repeat=10):
        G = nx.DiGraph()
        G.add_nodes_from(vertices)
        
        # Constrói o grafo com base no estado atual
        for (u, v), estado in zip(pares, estados):
            if estado == 1:
                G.add_edge(u, v)
            elif estado == 2:
                G.add_edge(v, u)
                
        # FILTRO 1: É Acíclico? (Rejeita se tiver ciclos)
        if not nx.is_directed_acyclic_graph(G):
            continue
            
        # FILTRO 2: É Conexo? (Rejeita grafos partidos ou variáveis isoladas)
        # Usamos conectividade fraca porque as setas têm direção, 
        # mas queremos apenas que a "estrutura" seja unida.
        if not nx.is_weakly_connected(G):
            continue
            
        # --- GERANDO A FÓRMULA SEM ---
        linhas_formula = []
        
        for no in vertices:
            pais = list(G.predecessors(no))
            if len(pais) > 0:
                # É endógena (recebe pelo menos uma seta)
                linhas_formula.append(f"{no} ~ {' + '.join(pais)}")
            else:
                # É exógena pura (raiz dentro deste modelo)
                linhas_formula.append(f"{no} ~~ {no}")
                
        formula = "\n".join(linhas_formula)
        arestas = [f"{u}->{v}" for u, v in G.edges()]
        num_arestas = G.number_of_edges()
        
        # Nomeia o modelo indicando o ID e o número de arestas
        nome_modelo = f"M{contador:05d}_Edges{num_arestas}"
        
        dags_validas[nome_modelo] = {
            "formula": formula,
            "Arestas": arestas,
            "Num_Arestas": num_arestas
        }
        
        contador += 1
        
    return dags_validas

# --- Executando e Verificando ---
print("Gerando modelos... Isso pode levar alguns segundos.")
modelos_conexos = gerar_todas_dags_conexas_5v()

print(f"\nTotal de DAGs Conexas geradas: {len(modelos_conexos)}")

# Exibindo um modelo não-saturado aleatório (ex: o de índice 15000)
chave_exemplo = list(modelos_conexos.keys())[15000]
print(f"\nExemplo - {chave_exemplo}:")
print(modelos_conexos[chave_exemplo]["formula"])
print("Arestas:", modelos_conexos[chave_exemplo]["Arestas"])

Gerando modelos... Isso pode levar alguns segundos.

Total de DAGs Conexas geradas: 26430

Exemplo - M15001_Edges7:
Visual ~~ Visual
Somatomotor ~ Visual
Default_mode ~ Visual
Cerebellar ~ Visual + Somatomotor + Control
Control ~ Visual + Somatomotor
Arestas: ['Visual->Somatomotor', 'Visual->Default_mode', 'Visual->Cerebellar', 'Visual->Control', 'Somatomotor->Cerebellar', 'Somatomotor->Control', 'Control->Cerebellar']


In [5]:
with open(salvar_formulas, "w", encoding="utf-8") as file:
    json.dump(modelos_conexos, file, indent=4)

### Ajuste de todos os modelos nos dois conjuntos.

In [6]:
def roda_os_modelos(df, modelos, retornar_inspection=False):    
    df_resultados = pd.DataFrame()
    dicionario = dict()

    for nome, modelo in modelos.items():
        formula = modelo["formula"]
        model = semopy.Model(formula)
        model.fit(df, obj="MLW")
        stats = semopy.stats.calc_stats(model)
        df_resultados = pd.concat([df_resultados, stats])
        if retornar_inspection:
            edges_inspection = model.inspect()
            dicionario[nome] = edges_inspection
            
   
    df_resultados["Modelo"] = modelos.keys()
    df_resultados.set_index("Modelo", inplace=True)

    if retornar_inspection:
        return df_resultados, dicionario

    return df_resultados

#### 1) Conjunto de Controle

In [7]:
df_resultados_controle = roda_os_modelos(df_controle, modelos_conexos)
df_resultados_controle

,DoF,DoF Baseline,chi2,chi2 p-value,chi2 Baseline,CFI,GFI,AGFI,NFI,TLI,RMSEA,AIC,BIC,LogLik
Modelo,,,,,,,,,,,,,,
M00001_Edges4,6,10,0.000021,1.000000e+00,563.068601,1.010849,1.000000,1.000000,1.000000,1.018081,0.000000,18.000000,56.438896,3.875706e-08
M00002_Edges4,6,10,73.790841,6.805667e-14,563.068639,0.877428,0.868949,0.781581,0.868949,0.795713,0.146283,17.721018,56.159914,1.394912e-01
M00003_Edges4,6,10,49.340974,6.371482e-09,563.068639,0.921635,0.912371,0.853952,0.912371,0.869392,0.116965,17.813456,56.252352,9.327216e-02
M00004_Edges4,6,10,119.283452,0.000000e+00,563.068677,0.795173,0.788155,0.646925,0.788155,0.658622,0.189100,17.549023,55.987919,2.254886e-01
M00005_Edges4,6,10,175.959190,0.000000e+00,563.068639,0.692698,0.687500,0.479166,0.687500,0.487830,0.231622,17.334748,55.773644,3.326261e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
M26426_Edges9,1,10,49.116887,2.411515e-12,563.068714,0.913000,0.912769,0.127693,0.912769,0.130002,0.301878,27.814303,87.608141,9.284856e-02
M26427_Edges10,0,10,0.000084,NaN,563.068714,1.000000,1.000000,NaN,1.000000,NaN,inf,30.000000,94.064826,1.588271e-07
M26428_Edges9,1,10,0.000136,9.906839e-01,563.068676,1.001808,1.000000,0.999998,1.000000,1.018078,0.000000,27.999999,87.793838,2.577220e-07


In [8]:
df_resultados_controle.to_csv(salvar_modelos_controle, index=True)

#### 2) Conjunto autismo

In [10]:
df_resultados_autismo = roda_os_modelos(df_autismo, modelos_conexos)
df_resultados_autismo

,DoF,DoF Baseline,chi2,chi2 p-value,chi2 Baseline,CFI,GFI,AGFI,NFI,TLI,RMSEA,AIC,BIC,LogLik
Modelo,,,,,,,,,,,,,,
M00001_Edges4,6,10,0.000018,1.000000e+00,536.344796,1.011399,1.000000,1.000000,1.000000,1.018999,0.000000,18.000000,50.872402,6.458758e-08
M00002_Edges4,6,10,54.289245,6.449842e-10,536.344817,0.908255,0.898779,0.831299,0.898779,0.847092,0.168341,17.619023,50.491425,1.904886e-01
M00003_Edges4,6,10,54.475540,5.915098e-10,536.344816,0.907902,0.898432,0.830720,0.898432,0.846503,0.168666,17.617716,50.490118,1.911422e-01
M00004_Edges4,6,10,99.262776,0.000000e+00,536.344837,0.822811,0.814927,0.691546,0.814927,0.704684,0.233948,17.303419,50.175822,3.482904e-01
M00005_Edges4,6,10,139.081771,0.000000e+00,536.344817,0.747159,0.740686,0.567810,0.740686,0.578598,0.279463,17.023988,49.896390,4.880062e-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
M26426_Edges9,1,10,39.499963,3.280657e-10,536.344857,0.926854,0.926353,0.263534,0.926353,0.268541,0.368189,27.722807,78.857656,1.385964e-01
M26427_Edges10,0,10,0.000193,NaN,536.344857,1.000000,1.000000,NaN,1.000000,NaN,inf,29.999999,84.787336,6.785909e-07
M26428_Edges9,1,10,0.000037,9.951508e-01,536.344837,1.001900,1.000000,0.999999,1.000000,1.018998,0.000000,28.000000,79.134848,1.296069e-07


In [11]:
df_resultados_autismo.to_csv(salvar_modelos_autismo, index=True)